# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/imaniftikhar/week1_flyrank/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Primary Research Question:** Can content staleness (days_since_update) and historical traffic scale (log_peak_clicks) reliably predict organic search traffic decay (target_click_loss) across unseen client domains?

**Decision Supported:** Directional prioritization for editorial teams—identifying high-impact content refresh candidates to prevent unnecessary full-page rewrites and optimize content maintenance resource allocation.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [1]:
import os
import getpass
import numpy as np
import pandas as pd
import duckdb
import lightgbm as lgb
from huggingface_hub import hf_hub_download

# 1. Download Parquet warehouse tables
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste Hugging Face Token (hf_...): ')
token_str = HF_TOKEN.strip()
repo_id = "FlyRank/internship-warehouse"

fact_path = hf_hub_download(repo_id=repo_id, filename="fact_content_daily_performance_sample.parquet", repo_type="dataset", token=token_str)
dim_path = hf_hub_download(repo_id=repo_id, filename="dim_content.parquet", repo_type="dataset", token=token_str)

con = duckdb.connect()

# 2. Query Lane 2 Content Performance Features
query = f"""
WITH aggregated_performance AS (
    SELECT
        content_hash_id,
        SUM(CASE WHEN report_date >= CURRENT_DATE - INTERVAL '30 days' THEN gsc_clicks ELSE 0 END) AS clicks_last_30d,
        MAX(gsc_clicks) AS peak_clicks_30d
    FROM read_parquet('{fact_path}')
    GROUP BY content_hash_id
)
SELECT
    c.url_hash_id,
    c.client_hash_id,
    c.word_count,
    DATE_DIFF('day', c.content_updated_date::DATE, CURRENT_DATE) AS days_since_update,
    COALESCE(p.clicks_last_30d, 0) AS clicks_last_30d,
    COALESCE(p.peak_clicks_30d, 0) AS peak_clicks_30d
FROM read_parquet('{dim_path}') c
LEFT JOIN aggregated_performance p ON c.content_hash_id = p.content_hash_id
WHERE c.url_hash_id IS NOT NULL
"""

df = con.execute(query).df()

# Exclude inactive pages with no historical search traffic
df = df[df['peak_clicks_30d'] >= 10].copy()
df['word_count'] = df['word_count'].fillna(0)

print(f"Dataset Ingested: {len(df)} active URLs across {df['client_hash_id'].nunique()} client domains.")

Paste Hugging Face Token (hf_...): ··········


fact_content_daily_performance_sample.pa(…): reconstructing file:   0%|          |  0.00B /  145MB            

fact_content_daily_performance_sample.pa(…): downloading bytes:           |  0.00B            

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Dataset Ingested: 932 active URLs across 23 client domains.


**Data Release:** Loaded directly from the FlyRank/internship-warehouse dataset (fact_content_daily_performance_sample and dim_content).

**Filtering & Exclusions:** Filtered to pages with peak_clicks_30d >= 10 to eliminate low-traffic noise and unindexed cold-start URLs where decay cannot be evaluated.

**Privacy Controls:** All client identifiers (client_hash_id) and URLs (url_hash_id) are anonymized hashes.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [2]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Feature Engineering
df['log_peak_clicks'] = np.log1p(df['peak_clicks_30d'])
df['target_click_loss'] = (df['peak_clicks_30d'] - df['clicks_last_30d']).clip(lower=0)

features = ['days_since_update', 'log_peak_clicks', 'word_count']

# Grouped Split by client_hash_id (Honest Evaluation)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))

X_tr, y_tr = df.iloc[train_idx][features], df.iloc[train_idx]['target_click_loss']
X_te, y_te = df.iloc[test_idx][features], df.iloc[test_idx]['target_click_loss']

# Baseline Model: Predict Mean Loss
baseline_pred = np.full_like(y_te, y_tr.mean())

# Trained Model: LightGBM
model = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42, verbosity=-1)
model.fit(X_tr, y_tr)
model_pred = model.predict(X_te)

**Target Definition:** target_click_loss ($=\text{Peak Clicks} - \text{Current 30d Clicks}$), measuring absolute volume drop.

**Leakage Prevention:** Grouped split by client_hash_id guarantees zero client overlap between training and test sets, avoiding domain authority memorization. Using log_peak_clicks prevents mathematical identity correlation.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [3]:
mae_base = mean_absolute_error(y_te, baseline_pred)
rmse_base = np.sqrt(mean_squared_error(y_te, baseline_pred))

mae_model = mean_absolute_error(y_te, model_pred)
rmse_model = np.sqrt(mean_squared_error(y_te, model_pred))

results_df = pd.DataFrame({
    'Model': ['Naive Mean Baseline', 'LightGBM (Grouped Split)'],
    'MAE': [mae_base, mae_model],
    'RMSE': [rmse_base, rmse_model]
})

print("=== Honest Validation Performance ===")
print(results_df.to_string(index=False))

=== Honest Validation Performance ===
                   Model      MAE      RMSE
     Naive Mean Baseline 9.720070 26.383126
LightGBM (Grouped Split) 5.511051 21.721572


## 5. Limitations

*What this work cannot claim.*

**Non-Causal Association:** The model estimates decay patterns based on observational staleness and traffic scale. Predicted click loss indicates directional risk for prioritization, not a guaranteed $+24\%$ traffic lift upon updating.

**SERP Feature Shifts:** The model cannot detect external layout changes (e.g., emergence of AI Overviews or direct answer boxes) that systematically reduce click-through rates across entire search topics regardless of content freshness.

**Cold-Start Constraint:** Requires historical search traffic (peak_clicks_30d >= 10); unindexed pages or newly onboarded domains with less than 30 days of telemetry cannot be evaluated.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [4]:
# Apply model across dataset to build the action queue
df['predicted_click_loss'] = model.predict(df[features])

def assign_action_and_reason(row):
    loss = row['predicted_click_loss']
    staleness = row['days_since_update']
    words = row['word_count']

    if loss > 15 and staleness > 40:
        return 'PRIORITY_REFRESH', 'STALE_CONTENT_HIGH_LOSS'
    elif loss > 15 and staleness <= 40:
        return 'TECHNICAL_AUDIT', 'RECENT_UPDATE_HIGH_LOSS'
    elif staleness > 365 and words < 400:
        return 'PRUNE_OR_CONSOLIDATE', 'LOW_TRAFFIC_THIN_CONTENT'
    else:
        return 'MONITOR', 'STABLE_PERFORMANCE'

df[['action', 'reason_code']] = df.apply(assign_action_and_reason, axis=1, result_type='expand')
action_queue = df.sort_values(by='predicted_click_loss', ascending=False)

print("=== Top 10 Priority Content Action Queue ===")
print(action_queue[['url_hash_id', 'days_since_update', 'peak_clicks_30d', 'predicted_click_loss', 'action', 'reason_code']].head(10).to_string(index=False))

=== Top 10 Priority Content Action Queue ===
         url_hash_id  days_since_update  peak_clicks_30d  predicted_click_loss           action             reason_code
url_464ff2db610e43bc                 59              890            172.990232 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_226c0d14bd4d70a5                 69              345            151.409816 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_a120b548329280b6                 57              329            144.022933 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_8b5c7405c6a7118f                 57              232            144.022933 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_b71a266de03ec98e                 55               68            144.022933 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_8b0a34daf5bac290                 59               49            144.022933 PRIORITY_REFRESH STALE_CONTENT_HIGH_LOSS
url_f37479058155ca8e                 57               98            144.022933 PRIORITY_REFRESH STALE_CONTENT_HIGH_

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [5]:
import os

# 1. Create output directory
os.makedirs('work/outputs', exist_ok=True)

# 2. Export Action Queue CSV
queue_path = 'work/outputs/capstone_content_action_queue.csv'
action_queue.to_csv(queue_path, index=False)

# 3. Export Summary Statistics CSV
summary_df = action_queue['action'].value_counts().reset_index()
summary_df.columns = ['Action Category', 'URL Count']
summary_path = 'work/outputs/capstone_action_summary_stats.csv'
summary_df.to_csv(summary_path, index=False)

# 4. Export Model Performance Summary CSV
results_df.to_csv('work/outputs/capstone_model_performance.csv', index=False)

print("Artifacts generated and saved to work/outputs/:")
print(f" - {queue_path}")
print(f" - {summary_path}")
print(f" - work/outputs/capstone_model_performance.csv")
print("\n=== Action Queue Summary Statistics ===")
print(summary_df.to_string(index=False))

Artifacts generated and saved to work/outputs/:
 - work/outputs/capstone_content_action_queue.csv
 - work/outputs/capstone_action_summary_stats.csv
 - work/outputs/capstone_model_performance.csv

=== Action Queue Summary Statistics ===
 Action Category  URL Count
         MONITOR        566
PRIORITY_REFRESH        366


- *capstone_content_action_queue.csv*: Full list of prioritized URLs containing predicted traffic loss, assigned actions, and reason codes for editorial workflows.

- *capstone_action_summary_stats.csv:* Macro-level breakdown of content across the four action categories (PRIORITY_REFRESH, TECHNICAL_AUDIT, PRUNE_OR_CONSOLIDATE, MONITOR).

- *capstone_model_performance.csv:*  MAE and RMSE benchmarking metrics comparing LightGBM against the naive baseline under honest grouped splitting.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.